# 🎯 Memory-Safe Smart Undersampling for XGBoost

Bu notebook, **88 milyon satırlık** büyük bir veri setinden XGBoost eğitimi için dengeli bir veri seti oluşturur.

## ⚠️ Memory-Safe Approach

**ASLA** büyük negatif grupları `collect()` ile RAM'e yüklemeden önce örnekleme yapıyoruz!

```python
# ❌ YANLIŞ - RAM'i çökertir!
df = lf.collect().sample(n=1000)

# ✅ DOĞRU - Memory-safe
total = lf.select(pl.len()).collect().item()  # Sadece sayı alır
fraction = target_count / total
df = lf.sample(fraction=fraction).collect()   # LazyFrame üzerinde sample
```

## Sampling Strategy

| Grup | Filtre | Hedef Sayı |
|------|--------|------------|
| **Positives** | `bolus > 0` | TÜMÜ |
| **Hard Negatives** | `bolus == 0` AND `CGM > 180` | 2 × Positives |
| **Easy Negatives** | `bolus == 0` AND `CGM <= 180` | 1 × Positives |

---

## 🔗 1. Google Drive'a Bağlan

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive başarıyla bağlandı!")

## 📦 2. Setup - Kütüphaneleri Yükle

In [ ]:
# Polars kurulumu (Colab için)
!pip install polars -q

In [ ]:
import polars as pl
import time
import os

print(f"✅ Polars version: {pl.__version__}")

## 📂 3. Dosya Yollarını ve Parametreleri Tanımla

In [ ]:
# ==============================================================================
# KONFİGÜRASYON
# ==============================================================================

# Base path - Drive'daki proje klasörü
BASE_PATH = "/content/drive/MyDrive/DiabetesKD5/data/processed"

# Input ve Output dosyaları
INPUT_FILE = f"{BASE_PATH}/metabonet_processed.parquet"
OUTPUT_FILE = f"{BASE_PATH}/train_balanced_xgboost.parquet"

# Sampling parametreleri
RANDOM_SEED = 42
HARD_NEGATIVE_RATIO = 2  # Hard negatives = 2 × positives
EASY_NEGATIVE_RATIO = 1  # Easy negatives = 1 × positives

print("=" * 70)
print("🎯 Memory-Safe Smart Undersampling for XGBoost")
print("=" * 70)
print(f"\n📂 Input:  {INPUT_FILE}")
print(f"📂 Output: {OUTPUT_FILE}")
print(f"\n🎲 Random Seed: {RANDOM_SEED}")
print(f"📊 Hard Negative Ratio: {HARD_NEGATIVE_RATIO}x positives")
print(f"📊 Easy Negative Ratio: {EASY_NEGATIVE_RATIO}x positives")

In [ ]:
# Dosyanın var olup olmadığını kontrol et
if os.path.exists(INPUT_FILE):
    file_size_gb = os.path.getsize(INPUT_FILE) / (1024**3)
    print(f"✅ Input dosyası bulundu: {file_size_gb:.2f} GB")
else:
    raise FileNotFoundError(f"❌ HATA: Input dosyası bulunamadı: {INPUT_FILE}")

## 📖 4. Lazy Loading - LazyFrame Oluştur

In [ ]:
print("📖 Lazy DataFrames oluşturuluyor...")

start_time = time.time()

# Ana LazyFrame - veriyi belleğe YÜKLEMEZ
lf = pl.scan_parquet(INPUT_FILE)

# Her grup için lazy filtreler tanımla
lf_positives = lf.filter(pl.col("bolus") > 0)
lf_hard_negatives = lf.filter((pl.col("bolus") == 0) & (pl.col("CGM") > 180))
lf_easy_negatives = lf.filter((pl.col("bolus") == 0) & (pl.col("CGM") <= 180))

print("✅ Lazy DataFrames oluşturuldu (henüz veri yüklenmedi)")

## 📊 5. Grup Sayılarını Hesapla (Memory-Safe)

In [ ]:
# ==============================================================================
# GRUP SAYILARINI HESAPLA - Sadece count alır, veriyi yüklemez!
# ==============================================================================

print("-" * 70)
print("📊 Grup sayıları hesaplanıyor (memory-safe count)...")
print("-" * 70)

# Positives sayısını al
n_positives_total = lf_positives.select(pl.len()).collect().item()
print(f"✅ Positives (bolus > 0):           {n_positives_total:>12,} satır")

# Hard negatives sayısını al
n_hard_negatives_total = lf_hard_negatives.select(pl.len()).collect().item()
print(f"✅ Hard Negatives (CGM > 180):      {n_hard_negatives_total:>12,} satır")

# Easy negatives sayısını al
n_easy_negatives_total = lf_easy_negatives.select(pl.len()).collect().item()
print(f"✅ Easy Negatives (CGM ≤ 180):      {n_easy_negatives_total:>12,} satır")

print(f"\n📈 Toplam satır: {n_positives_total + n_hard_negatives_total + n_easy_negatives_total:,}")

## 🧮 6. Sampling Fractions Hesapla

In [ ]:
# ==============================================================================
# HEDEF SAYILARI VE FRACTION HESAPLA
# ==============================================================================

print("-" * 70)
print("🧮 Sampling fractions hesaplanıyor...")
print("-" * 70)

# Hedef sayıları hesapla
target_hard_negatives = n_positives_total * HARD_NEGATIVE_RATIO
target_easy_negatives = n_positives_total * EASY_NEGATIVE_RATIO

# Fraction hesapla (hedef / toplam)
# Eğer hedef > toplam ise, fraction = 1.0 (tümünü al)
fraction_hard = min(target_hard_negatives / n_hard_negatives_total, 1.0)
fraction_easy = min(target_easy_negatives / n_easy_negatives_total, 1.0)

print(f"\n🎯 Hedef Hard Negatives: {target_hard_negatives:,} ({HARD_NEGATIVE_RATIO}x positives)")
print(f"   → Fraction: {fraction_hard:.6f} ({fraction_hard*100:.4f}%)")

print(f"\n🎯 Hedef Easy Negatives: {target_easy_negatives:,} ({EASY_NEGATIVE_RATIO}x positives)")
print(f"   → Fraction: {fraction_easy:.6f} ({fraction_easy*100:.4f}%)")

## 📥 7. Memory-Safe Sampling & Collection

In [ ]:
# ==============================================================================
# POSITIVES - Tümünü topla (küçük grup, RAM'e sığar)
# ==============================================================================

print("-" * 70)
print("📥 [1/3] Positives (bolus > 0) toplanıyor...")
print("-" * 70)

df_positives = lf_positives.collect()
n_positives = len(df_positives)

print(f"✅ Positives toplandı: {n_positives:,} satır")

In [ ]:
# ==============================================================================
# HARD NEGATIVES - LazyFrame üzerinde sample, sonra collect
# ==============================================================================

print("-" * 70)
print("📥 [2/3] Hard Negatives örnekleniyor (memory-safe)...")
print("-" * 70)

# ⚠️ CRITICAL: sample() ÖNCE LazyFrame üzerinde uygulanıyor!
df_hard_negatives = (
    lf_hard_negatives
    .sample(fraction=fraction_hard, seed=RANDOM_SEED, shuffle=True)
    .collect()
)
n_hard_negatives = len(df_hard_negatives)

print(f"✅ Hard Negatives örneklendi: {n_hard_negatives:,} satır")
print(f"   (Hedef: {target_hard_negatives:,}, Sapma: {n_hard_negatives - target_hard_negatives:+,})")

In [ ]:
# ==============================================================================
# EASY NEGATIVES - LazyFrame üzerinde sample, sonra collect
# ==============================================================================

print("-" * 70)
print("📥 [3/3] Easy Negatives örnekleniyor (memory-safe)...")
print("-" * 70)

# ⚠️ CRITICAL: sample() ÖNCE LazyFrame üzerinde uygulanıyor!
df_easy_negatives = (
    lf_easy_negatives
    .sample(fraction=fraction_easy, seed=RANDOM_SEED, shuffle=True)
    .collect()
)
n_easy_negatives = len(df_easy_negatives)

print(f"✅ Easy Negatives örneklendi: {n_easy_negatives:,} satır")
print(f"   (Hedef: {target_easy_negatives:,}, Sapma: {n_easy_negatives - target_easy_negatives:+,})")

## 🔀 8. Birleştir ve Karıştır

In [ ]:
print("-" * 70)
print("🔀 DataFrame'ler birleştiriliyor ve karıştırılıyor...")
print("-" * 70)

# Üç DataFrame'i birleştir
df_combined = pl.concat([
    df_positives,
    df_hard_negatives,
    df_easy_negatives
])

print(f"📊 Birleştirilmiş veri seti boyutu: {len(df_combined):,} satır")

# Veri setini karıştır
df_final = df_combined.sample(
    fraction=1.0,
    shuffle=True,
    seed=RANDOM_SEED
)

print("✅ Veri seti başarıyla karıştırıldı")

## 💾 9. Parquet Olarak Kaydet

In [ ]:
print("-" * 70)
print("💾 Parquet dosyasına kaydediliyor...")
print("-" * 70)

df_final.write_parquet(OUTPUT_FILE)

elapsed_time = time.time() - start_time

# Dosya boyutunu kontrol et
output_size_mb = os.path.getsize(OUTPUT_FILE) / (1024**2)

print(f"✅ Kaydedildi: {OUTPUT_FILE}")
print(f"📦 Dosya boyutu: {output_size_mb:.2f} MB")
print(f"⏱️  Toplam işlem süresi: {elapsed_time:.2f} saniye")

## 📋 10. Final Rapor

In [ ]:
# ==============================================================================
# FINAL RAPOR
# ==============================================================================

total_rows = len(df_final)
n_positive_final = df_final.filter(pl.col("bolus") > 0).height
n_negative_final = total_rows - n_positive_final
class_balance = (n_positive_final / total_rows) * 100

print("\n" + "=" * 70)
print("📋 FINAL RAPOR - XGBoost için Smart Undersampling Tamamlandı")
print("=" * 70)

print(f"""
┌───────────────────────────────────────────────────────────────────┐
│  ÖRNEKLEME SONUÇLARI                                              │
├───────────────────────────────────────────────────────────────────┤
│  ✓ Positives (bolus > 0):           {n_positives:>12,} satır (TÜMÜ)   │
│  ✓ Hard Negatives (CGM > 180):      {n_hard_negatives:>12,} satır (2x)    │
│  ✓ Easy Negatives (CGM ≤ 180):      {n_easy_negatives:>12,} satır (1x)    │
├───────────────────────────────────────────────────────────────────┤
│  📊 TOPLAM VERİ SETİ BOYUTU:        {total_rows:>12,} satır          │
└───────────────────────────────────────────────────────────────────┘

┌───────────────────────────────────────────────────────────────────┐
│  SINIF DAĞILIMI                                                   │
├───────────────────────────────────────────────────────────────────┤
│  Pozitif Sınıf (bolus > 0):         {class_balance:>12.2f}%              │
│  Negatif Sınıf (bolus = 0):         {100 - class_balance:>12.2f}%              │
├───────────────────────────────────────────────────────────────────┤
│  Pozitif : Negatif Oranı:           1 : {n_negative_final/n_positive_final:.2f}                    │
└───────────────────────────────────────────────────────────────────┘

📈 Orijinal dengesizlik: ~%3 pozitif (1:32 oran)
📈 Yeni dengeli oran: %{class_balance:.2f} pozitif (1:{n_negative_final/n_positive_final:.1f} oran)

🎯 XGBoost eğitimi için hazır!
""")

## 🧹 11. Bellek Temizliği

In [ ]:
# Eğitim başlamadan önce belleği temizle
del df_positives, df_hard_negatives, df_easy_negatives, df_combined

import gc
gc.collect()

print("✅ Bellek temizlendi!")
print(f"\n📂 Dengeli veri seti hazır: {OUTPUT_FILE}")

---

## ✅ Sonraki Adım: XGBoost Eğitimi

```python
import polars as pl

# Dengeli veri setini yükle
df_train = pl.read_parquet("/content/drive/MyDrive/DiabetesKD5/data/processed/train_balanced_xgboost.parquet")

# Features ve target ayır
X = df_train.drop("bolus").to_numpy()
y = (df_train["bolus"] > 0).cast(pl.Int32).to_numpy()

# XGBoost eğitimi...
```